### Module 6 : LLM Integration

#### Objective

Generate answers using retrieved Basel III context.

Learning Outcomes

- Connect an LLM
- Understand prompt engineering
- Build a RAG pipeline
- Generate grounded responses

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [5]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.load_local(
    "../vectorstore/faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

C:\Users\shrab\AppData\Local\Temp\ipykernel_17552\1213405182.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
C:\Users\shrab\AppData\Local\Temp\ipykernel_17552\1213405182.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [41]:
query = "What is Common Equity Tier 1?"

docs = vectorstore.similarity_search(
    query,
    k=4
)

In [42]:
context = "\n\n".join(
    doc.page_content
    for doc in docs
)

In [43]:
print(context[:1500])

 Regulatory adjustments applied in the calculation of Common Equity Tier 1 
Retained earnings and other comprehensive income include interim profit or loss. National 
authorities may consider appropriate audit, verification or review procedures. Dividends are 
removed from Common Equity Tier 1 in accordance with applicable accounting standards. 
The treatment of minority interest and the regulatory adjustments applied in the calculation of 
Common Equity Tier 1 are addressed in separate sections. 
Common shares issued by the bank 
53.  For an instrument to be included in Common Equity Tier 1 capital it must meet all of 
the criteria t
hat follow. The vast majority of internationally active banks are structured as joint 
stock companies11 and for these banks the criteria must be met solely with common shares. 
In the rare cases where banks need to issue non-voting common shares as part of Common 
Equity Tier 1, they must be identical to voting common shares of the issuing bank in all



### prompt Engineering

In [44]:
from langchain_core.prompts import ChatPromptTemplate

In [45]:
prompt = ChatPromptTemplate.from_template(
"""
You are an expert Basel III regulatory assistant.

Rules:

1. Answer ONLY from the provided context.
2. Never use outside knowledge.
3. If the answer is unavailable, say:
   "I don't have enough information in the provided Basel document."
4. Quote important regulatory terminology exactly.
5. Keep the answer concise.
6. Mention the relevant source pages at the end.

Context:
{context}

Question:
{question}

Answer:
"""
)

In [46]:
formatted_prompt = prompt.format(
    context=context,
    question=query
)

print(formatted_prompt)

Human: 
You are an expert Basel III regulatory assistant.

Rules:

1. Answer ONLY from the provided context.
2. Never use outside knowledge.
3. If the answer is unavailable, say:
   "I don't have enough information in the provided Basel document."
4. Quote important regulatory terminology exactly.
5. Keep the answer concise.
6. Mention the relevant source pages at the end.

Context:
 Regulatory adjustments applied in the calculation of Common Equity Tier 1 
Retained earnings and other comprehensive income include interim profit or loss. National 
authorities may consider appropriate audit, verification or review procedures. Dividends are 
removed from Common Equity Tier 1 in accordance with applicable accounting standards. 
The treatment of minority interest and the regulatory adjustments applied in the calculation of 
Common Equity Tier 1 are addressed in separate sections. 
Common shares issued by the bank 
53.  For an instrument to be included in Common Equity Tier 1 capital it mus

In [47]:
response = llm.invoke(formatted_prompt)

print(response.content)

Common Equity Tier 1 capital primarily consists of "common shares issued by the bank". It also includes "retained earnings and other comprehensive income", which encompasses "interim profit or loss", and is subject to "regulatory adjustments".

(Source: Regulatory adjustments applied in the calculation of Common Equity Tier 1; Common shares issued by the bank, paragraph 53)


In [48]:
sources = []

for doc in docs:
    page = doc.metadata["page"] + 1
    sources.append(page)

sources = sorted(set(sources))    

In [49]:
print("="*80)
print("QUESTION")
print("="*80)
print(query)

print()

print("="*80)
print("ANSWER")
print("="*80)
print(response.content)

print()

print("="*80)
print("SOURCE PAGES")
print("="*80)
print(sources)

QUESTION
What is Common Equity Tier 1?

ANSWER
Common Equity Tier 1 capital primarily consists of "common shares issued by the bank". It also includes "retained earnings and other comprehensive income", which encompasses "interim profit or loss", and is subject to "regulatory adjustments".

(Source: Regulatory adjustments applied in the calculation of Common Equity Tier 1; Common shares issued by the bank, paragraph 53)

SOURCE PAGES
[20, 21, 28, 67]
